# YOLOv8 WBC 탐지 모델 학습 (Colab용)

## 사용 전 필수 확인
- 런타임 > 런타임 유형 변경 > **T4 GPU** (또는 A100) 선택
- 셀을 **위에서 아래로 순서대로** 실행하세요

## 파이프라인 요약
```
공개 WBC 데이터셋(BCCD) 다운로드
  → Pascal VOC XML → YOLO 형식 변환
  → YOLOv8n 학습 (WBC 탐지 전용)
  → best.pt → wbc_detector.pt 로 저장 (구글 드라이브)
```
학습 완료 후 `wbc_detector.pt` 를 로컬 `PB_Smear/` 폴더에 넣으면 `app.py` 가 자동으로 인식합니다.

In [ ]:
# =============================================
# [Step 1] 구글 드라이브 마운트
# =============================================
from google.colab import drive
drive.mount('/content/drive')

# 결과물 저장 경로 (드라이브 내 자유롭게 변경 가능)
SAVE_DIR = '/content/drive/MyDrive/PB_Smear'

import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'저장 경로: {SAVE_DIR}')

In [ ]:
# =============================================
# [Step 2] 패키지 설치
# =============================================
!pip install ultralytics -q

import ultralytics
ultralytics.checks()   # GPU 인식 여부 확인

In [ ]:
# =============================================
# [Step 3] BCCD 공개 데이터셋 다운로드
#
# BCCD(Blood Cell Count Dataset): Pascal VOC XML 형식
# 클래스: WBC / RBC / Platelet
# 우리는 WBC 만 탐지하도록 필터링 예정
# =============================================
!git clone https://github.com/Shenggan/BCCD_Dataset.git /content/BCCD_Dataset -q
print('BCCD 데이터셋 다운로드 완료')

import glob
xml_files = glob.glob('/content/BCCD_Dataset/BCCD/Annotations/*.xml')
img_files = glob.glob('/content/BCCD_Dataset/BCCD/JPEGImages/*.jpg')
print(f'  XML 파일: {len(xml_files)}개')
print(f'  이미지:   {len(img_files)}개')

In [ ]:
# =============================================
# [Step 4] Pascal VOC XML → YOLO 형식(.txt) 변환
#
# YOLO 형식: 한 줄 = <class_id> <cx> <cy> <w> <h>  (모두 0~1 정규화)
# 여기서는 WBC만 class 0 으로 변환, RBC·Platelet 은 무시
# =============================================
import xml.etree.ElementTree as ET
import shutil

# YOLO 데이터셋 폴더 구조 생성
for split in ['train', 'val']:
    os.makedirs(f'/content/wbc_yolo/{split}/images', exist_ok=True)
    os.makedirs(f'/content/wbc_yolo/{split}/labels', exist_ok=True)

# WBC 만 탐지 대상 (class_id = 0)
TARGET_CLASS = 'WBC'


def voc_to_yolo(xml_path):
    """XML 1개를 파싱해 YOLO 라벨 행 리스트를 반환. WBC 외 클래스는 건너뜀."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_w = int(root.find('size/width').text)
    img_h = int(root.find('size/height').text)

    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.upper()
        if name != TARGET_CLASS:
            continue  # RBC, Platelet 무시

        bnd = obj.find('bndbox')
        x1 = float(bnd.find('xmin').text)
        y1 = float(bnd.find('ymin').text)
        x2 = float(bnd.find('xmax').text)
        y2 = float(bnd.find('ymax').text)

        # 중심점·폭·높이를 이미지 크기로 정규화
        cx = (x1 + x2) / 2 / img_w
        cy = (y1 + y2) / 2 / img_h
        bw = (x2 - x1) / img_w
        bh = (y2 - y1) / img_h

        lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    return lines


# 전체 XML 변환 + 학습/검증 8:2 분할
import random
random.seed(42)

valid_pairs = []   # (xml_path, img_path) 중 WBC 박스가 1개 이상인 것만

for xml_path in xml_files:
    stem     = os.path.splitext(os.path.basename(xml_path))[0]
    img_path = f'/content/BCCD_Dataset/BCCD/JPEGImages/{stem}.jpg'
    if not os.path.exists(img_path):
        continue
    lines = voc_to_yolo(xml_path)
    if lines:   # WBC 박스가 있는 이미지만 사용
        valid_pairs.append((xml_path, img_path, lines))

random.shuffle(valid_pairs)
split_idx = int(len(valid_pairs) * 0.8)
train_pairs = valid_pairs[:split_idx]
val_pairs   = valid_pairs[split_idx:]

def copy_pair(pairs, split):
    for xml_path, img_path, lines in pairs:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        shutil.copy(img_path, f'/content/wbc_yolo/{split}/images/{stem}.jpg')
        with open(f'/content/wbc_yolo/{split}/labels/{stem}.txt', 'w') as f:
            f.write('\n'.join(lines))

copy_pair(train_pairs, 'train')
copy_pair(val_pairs,   'val')

print(f'변환 완료 — 학습: {len(train_pairs)}장 / 검증: {len(val_pairs)}장')

In [ ]:
# =============================================
# [Step 5] data.yaml 생성 (YOLO 학습 설정 파일)
# =============================================
yaml_content = """\
path: /content/wbc_yolo
train: train/images
val:   val/images

nc: 1
names: ['WBC']
"""

with open('/content/wbc_yolo/data.yaml', 'w') as f:
    f.write(yaml_content)

print('data.yaml 생성 완료')
print(yaml_content)

In [ ]:
# =============================================
# [Step 6] YOLOv8n 학습
#
# yolov8n.pt : nano 모델 (가장 가벼움, Colab 무료 GPU에 적합)
# imgsz=640  : 입력 해상도
# epochs=80  : 데이터가 적으므로 충분한 에포크
# batch=16   : T4 GPU 메모리에 맞춤
# =============================================
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # 사전학습 가중치 자동 다운로드

results = model.train(
    data='/content/wbc_yolo/data.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    patience=15,          # 15 에포크 동안 개선 없으면 조기 종료
    device=0,             # GPU 0번 사용
    project='/content/runs',
    name='wbc_detector',
    exist_ok=True,
    verbose=True,
)

print('학습 완료!')

In [ ]:
# =============================================
# [Step 7] 검증 지표 확인
# =============================================
metrics = model.val()
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")

In [ ]:
# =============================================
# [Step 8] 학습 결과 시각화
# =============================================
from IPython.display import Image as IPImage, display

results_dir = '/content/runs/wbc_detector'

# 학습 곡선 (loss / mAP)
results_png = os.path.join(results_dir, 'results.png')
if os.path.exists(results_png):
    display(IPImage(results_png, width=900))

# 검증 배치 예측 미리보기
val_pred = os.path.join(results_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_pred):
    display(IPImage(val_pred, width=600))

In [ ]:
# =============================================
# [Step 9] 최종 모델을 구글 드라이브에 저장
#
# 저장 완료 후:
#   드라이브에서 wbc_detector.pt 를 다운로드 →
#   로컬 PB_Smear/ 폴더에 넣기 →
#   python app.py 재실행
# =============================================
import shutil

best_pt_src  = '/content/runs/wbc_detector/weights/best.pt'
best_pt_dst  = os.path.join(SAVE_DIR, 'wbc_detector.pt')

if os.path.exists(best_pt_src):
    shutil.copy(best_pt_src, best_pt_dst)
    print(f'저장 완료: {best_pt_dst}')
    print()
    print('다음 단계:')
    print('  1. 구글 드라이브 > PB_Smear > wbc_detector.pt 다운로드')
    print('  2. 맥북의 PB_Smear/ 폴더에 wbc_detector.pt 넣기')
    print('  3. 터미널에서: python app.py')
else:
    print('[ERROR] best.pt 를 찾지 못했습니다. 학습이 완료됐는지 확인하세요.')
    print(f'  예상 경로: {best_pt_src}')